# Programación declarativa @ URJC
# Programación funcional
## Examen de prueba 3 (estilo convocatoria extraordinaria)

La duración del examen es de 1h:20m.

> Las celdas marcadas con `// IMPLEMENTA TU RESPUESTA` son las que tienes que completar. Ejecuta los tests de cada apartado con la celda `run(...)` que le sigue.

# Preámbulo

In [ ]:
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

### Algunas definiciones de tipos y funciones auxiliares

In [ ]:
object Signatures:
    abstract class List[A]:

        // Common HOFs
        def foldRight[B](nil: B)(cons: (A, B) => B): B
        def foldLeft[B](initial: B)(update: (B, A) => B): B
        def map[B](f: A => B): List[B]
        def flatMap[B](f: A => List[B]): List[B]
        def filter(f: A => Boolean): List[A]
        def forall(pred: A => Boolean): Boolean
        def exists(pred: A => Boolean): Boolean

        // Common functions
        def length: Int
        def reverse: List[A]
        def drop(i: Int): List[A]
        def zip[B](other: List[B]): List[(A, B)]
        def appended(e: A): List[A]

    // List.fill(n)(e) crea una lista con n copias de e:  List.fill(3)('a') == List('a','a','a')

In [ ]:
// type Tree[A] = 1 + Tree[A] * A * Tree[A]

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:

    def void[A]: Tree[A] =
        Empty()

    def leaf[A](a: A): Node[A] =
        Node(Empty(), a, Empty())

    def right[A](a: A, tree: Tree[A]): Node[A] =
        Node(Empty(), a, tree)

    def left[A](tree: Tree[A], a: A): Node[A] =
        Node(tree, a, Empty())

    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] =
        Node(left, a, right)

import Tree._

### Definiciones auxiliares sobre la correspondencia Curry-Howard y Tipos Algebraicos de Datos

In [ ]:
trait Isomorphic[A, B]:

    def from(a: A): B

    def to(b: B): A

    // equality

    def equalA(a1: A, a2: A): Boolean =
        a1 == a2

    def equalB(b1: B, b2: B): Boolean =
        b1 == b2

    // Bijection laws

    def law1(a: A): Boolean =
        equalA(to(from(a)), a)

    def law2(b: B): Boolean =
        equalB(from(to(b)), b)

In [ ]:
type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

# Ejercicio 1
__(2 puntos)__

__a) (1 punto)__ Utiliza la correspondencia de Curry-Howard para demostrar la siguiente tautología de la lógica proposicional intuicionista:

##### $\vdash p \vee (q \wedge r) \rightarrow (p \vee q) \wedge (p \vee r)$

In [ ]:
// IMPLEMENTA TU RESPUESTA

def proof[P, Q, R]: Either[P, (Q, R)] => (Either[P, Q], Either[P, R]) =
    ???

__b) (1 punto)__ Se desea utilizar la correspondencia de Curry-Howard para demostrar la validez de la siguiente tautología de la lógica clásica proposicional:

##### $\vdash (\neg p \rightarrow q) \rightarrow p \vee q$

Para ello, utiliza como premisa adicional la ley del tercio excluso aplicada a la proposición $p$.

In [ ]:
// IMPLEMENTA TU RESPUESTA

def proof[P, Q](lem: ExcludedMiddle): (Not[P] => Q) => Either[P, Q] =
    ???

# Ejercicio 2
__(2 puntos)__

__a) (1 punto)__ Demuestra, mediante la equivalencia entre tipos algebraicos de datos y las operaciones aritméticas, y mediante una instanciación del trait `Isomorphic`, que para cualesquiera tipos `A` y `B` los tipos `Option[Either[A, B]]` y `Either[Option[A], B]` son isomorfos.

_Escribe aquí la cadena de igualdades:_

|Option[Either[A, B]]| =

...

|Either[Option[A], B]|

In [ ]:
// IMPLEMENTA TU RESPUESTA

class Iso[A, B] extends Isomorphic[Option[Either[A, B]], Either[Option[A], B]]:

    def from(o: Option[Either[A, B]]): Either[Option[A], B] =
        ???

    def to(e: Either[Option[A], B]): Option[Either[A, B]] =
        ???

In [ ]:
class TestIso(iso: Isomorphic[Option[Either[Int, Char]], Either[Option[Int], Char]])
extends AnyFlatSpec with should.Matchers:

    "law1" should "hold" in:
        iso.law1(None) shouldBe true
        iso.law1(Some(Left(3))) shouldBe true
        iso.law1(Some(Right('a'))) shouldBe true

    "law2" should "hold" in:
        iso.law2(Left(None)) shouldBe true
        iso.law2(Left(Some(3))) shouldBe true
        iso.law2(Right('a')) shouldBe true

In [ ]:
run(TestIso(Iso[Int, Char]()))

__b) (1 punto)__ Demuestra o refuta que, para cualquier tipo `A`, los tipos `Boolean => Option[A]` y `Option[(A, A)]` son isomorfos. Si no lo son, justifícalo comparando sus cardinalidades para un tipo `A` concreto.

_Escribe aquí tu respuesta:_

# Ejercicio 3
__(3 puntos)__

Se desea implementar la función `decodeRuns`, que recibe una lista de pares `(elemento, número de repeticiones)` y devuelve la lista obtenida al repetir cada elemento el número de veces indicado, respetando el orden. Por ejemplo, la función deberá satisfacer los siguientes tests:

In [ ]:
class TestDecodeRuns(decodeRuns: List[(Char, Int)] => List[Char])
extends AnyFlatSpec with should.Matchers:

    "decodeRuns" should "work" in:
        decodeRuns(List()) shouldBe List()
        decodeRuns(List(('a', 3))) shouldBe List('a', 'a', 'a')
        decodeRuns(List(('a', 2), ('b', 1), ('c', 3))) shouldBe List('a', 'a', 'b', 'c', 'c', 'c')
        decodeRuns(List(('x', 0), ('y', 2))) shouldBe List('y', 'y')

Para ello, se puede utilizar la función `List.fill` del preámbulo: `List.fill(n)(e)` crea una lista con `n` copias de `e` (y la lista vacía si `n` es 0).

__a) (1 punto)__ Implementa la función `decodeRuns` mediante recursividad (no final).

In [ ]:
// IMPLEMENTA TU RESPUESTA

def decodeRuns[A](l: List[(A, Int)]): List[A] =
    ???

In [ ]:
run(TestDecodeRuns(decodeRuns))

__b) (1 punto)__ Implementa la función `decodeRuns` mediante la función de orden superior `foldRight`.

In [ ]:
// IMPLEMENTA TU RESPUESTA

def decodeRuns[A](l: List[(A, Int)]): List[A] =
    ???

In [ ]:
run(TestDecodeRuns(decodeRuns))

__c) (1 punto)__ Implementa la función `decodeRuns` sin recursión, mediante la función de orden superior `flatMap`.

In [ ]:
// IMPLEMENTA TU RESPUESTA

def decodeRuns[A](l: List[(A, Int)]): List[A] =
    ???

In [ ]:
run(TestDecodeRuns(decodeRuns))

# Ejercicio 4
__(3 puntos)__

Considérense las siguientes funciones `countdown` y `digits`:

* `countdown(n)` devuelve la cuenta atrás desde `n` hasta `1`.
* `digits(n)` devuelve los dígitos de un número no negativo, empezando por el de las unidades (y la lista vacía para el `0`).

In [ ]:
def countdown(n: Int): List[Int] =
    if n <= 0 then Nil
    else n :: countdown(n - 1)

In [ ]:
def digits(n: Int): List[Int] =
    if n == 0 then Nil
    else (n % 10) :: digits(n / 10)

In [ ]:
class TestGen(countdown: Int => List[Int], digits: Int => List[Int])
extends AnyFlatSpec with should.Matchers:

    "countdown" should "work" in:
        countdown(0) shouldBe List()
        countdown(3) shouldBe List(3, 2, 1)

    "digits" should "work" in:
        digits(0) shouldBe List()
        digits(7) shouldBe List(7)
        digits(1234) shouldBe List(4, 3, 2, 1)

In [ ]:
run(TestGen(countdown, digits))

Ambas funciones generan una lista a partir de un valor inicial (la _semilla_): en cada paso, o bien se termina, o bien se produce un elemento de la lista y una nueva semilla con la que continuar.

__a) (1,5 puntos)__ Implementa la función de orden superior `unfold`, que encapsula este patrón. La función `next` devuelve `None` si hay que terminar, o `Some((elemento, nuevaSemilla))` en otro caso.

In [ ]:
// IMPLEMENTA TU RESPUESTA

def unfold[A, B](seed: B)(next: B => Option[(A, B)]): List[A] =
    ???

__b) (1 punto)__ Implementa de una manera más modular las funciones `countdown` y `digits` mediante la función `unfold`.

In [ ]:
// IMPLEMENTA TU RESPUESTA

def countdown(n: Int): List[Int] =
    ???

def digits(n: Int): List[Int] =
    ???

In [ ]:
run(TestGen(countdown, digits))

__c) (0,5 puntos)__ Implementa, mediante `unfold`, la función `range(from, to)`, que devuelve la lista de enteros desde `from` hasta `to`, ambos incluidos (vacía si `from > to`).

In [ ]:
class TestRange(range: (Int, Int) => List[Int]) extends AnyFlatSpec with should.Matchers:

    "range" should "work" in:
        range(1, 5) shouldBe List(1, 2, 3, 4, 5)
        range(3, 3) shouldBe List(3)
        range(4, 2) shouldBe List()

In [ ]:
// IMPLEMENTA TU RESPUESTA

def range(from: Int, to: Int): List[Int] =
    ???

In [ ]:
run(TestRange(range))